In [23]:
import os
import uuid
import shutil
#import pylatex
import pandas as pd
import numpy as np
import lightkurve as lk 
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

from datetime import datetime
from tqdm.notebook import tqdm
from multiprocessing import cpu_count, Pool
#from pylatex.section import Chapter, Subsection
#from pylatex import Document, Section, Figure, NoEscape, Command

%run ../pipeline.ipynb

import warnings
warnings.filterwarnings("ignore")

In [24]:
DOWNLOAD_PATH = "../data"
LIGHTCURVE_PATH = "../lightcurves"

In [25]:
lclist = pd.read_csv(
    "NIMO_lcnames.csv",
    dtype={
        "TIC": "string",
        "gaiadr3_source_id": "string",
    },
)
lclist.head()

,nimo_id,TIC,gaiadr3_source_id,group,num_sectors,Sector,name,lc_name
0,1,350559457,4767439021687332096,car,45,1,SPOC-120,TIC-350559457-1-SPOC-120
1,1,350559457,4767439021687332096,car,45,2,SPOC-120,TIC-350559457-2-SPOC-120
2,1,350559457,4767439021687332096,car,45,3,SPOC-120,TIC-350559457-3-SPOC-120
3,1,350559457,4767439021687332096,car,45,4,SPOC-120,TIC-350559457-4-SPOC-120
4,1,350559457,4767439021687332096,car,45,5,SPOC-120,TIC-350559457-5-SPOC-120


In [26]:
lclist_float = lclist.copy()
lclist_float['TIC'] = lclist_float['TIC'].astype('float')

In [27]:
lclist_float["epic_id"] = float("nan")
lclist_float["age_Myr"] = float("nan")
lclist_float["author"] = ""
lclist_float["year"] = 0

In [28]:
gaiatable=pd.read_csv('nimo_cleanedgaiadata.csv')

In [29]:
gaiatable[gaiatable['TIC'].notna()].head()

,nimo_ids_oid,nimo_id,TIC,gaiadr3_source_id,group,num_sectors,Sector,name,lc_name,solution_id,...,g_absmag,bp_absmag,rp_absmag,abs_g_rp,abs_bp_rp,abs_bp_g,source_table,g_erroverflux,bp_erroverflux,rp_erroverflux
0,98.0,28.0,50745567.0,3.216730e+18,thor,3.0,32.0,SPOC-120,TIC-50745567-32-SPOC-120,1636148068921376768,...,9.800984,11.698456,8.481298,1.319686,3.217158,1.897472,our table,NaN,NaN,NaN
1,284.0,28.0,50745567.0,3.216730e+18,thor,3.0,98.0,FFI-200s,TIC-50745567-98-FFI-200s,1636148068921376768,...,9.800984,11.698456,8.481298,1.319686,3.217158,1.897472,our table,NaN,NaN,NaN
2,24.0,28.0,50745567.0,3.216730e+18,thor,3.0,6.0,SPOC-120,TIC-50745567-6-SPOC-120,1636148068921376768,...,9.800984,11.698456,8.481298,1.319686,3.217158,1.897472,our table,NaN,NaN,NaN
3,256.0,11.0,158817023.0,3.319628e+18,thor,3.0,98.0,SPOC-120,TIC-158817023-98-SPOC-120,1636148068921376768,...,8.865663,10.312127,7.656003,1.209660,2.656124,1.446464,our table,NaN,NaN,NaN
4,8.0,11.0,158817023.0,3.319628e+18,thor,3.0,6.0,FFI-30min,TIC-158817023-6-FFI-30min,1636148068921376768,...,8.865663,10.312127,7.656003,1.209660,2.656124,1.446464,our table,NaN,NaN,NaN


In [30]:
gaiatable["author"] = ""
gaiatable["year"] = 0

In [31]:
#lclist = lclist[lclist["lc_name"] != "TIC-188172972-36-FFI-10min"].reset_index(drop=True)

In [32]:
# for index, currow in lclist.iterrows(): 
#     nimo_id=currow['nimo_id']
#     lcname=currow['lc_name']
    
#     if os.path.isfile(f'multiplots/{nimo_id}/{lcname}.png'): #don't replot ones that already have been plotted
#         continue

#     curlc=get_lc_string(lcname)
#     #print(curlc)
#     fig=multiplot3(curlc,lcname,gaiatable,lclist_float)


#     if not os.path.isdir(f'multiplots/{nimo_id}/'): #check if folder exists and if not make it
#         os.mkdir(f'multiplots/{nimo_id}/')

#     fig.savefig(f'multiplots/{nimo_id}/{lcname}.png',bbox_inches='tight')


In [34]:
for index, currow in lclist.iterrows(): 
    nimo_id=currow['nimo_id']
    lcname=currow['lc_name']
    
    if os.path.isfile(f'multiplots/{nimo_id}/{lcname}.png'): #don't replot ones that already have been plotted
        continue

    curlc=get_lc_string(lcname)
    #print(curlc)
    #fig=multiplot3(curlc,lcname,gaiatable,lclist)
    try: #make sure that an error with plotting one doesn't break the loop
        fig=multiplot3(curlc,lcname,gaiatable,lclist_float)

        if not os.path.isdir(f'multiplots/{nimo_id}/'): #check if folder exists and if not make it
            os.mkdir(f'multiplots/{nimo_id}/')

        fig.savefig(f'multiplots/{nimo_id}/{lcname}.png',bbox_inches='tight')
    except Exception as e:
        print(f'Error processing {lcname}: {e}')
        #print(curlc)

Error processing TIC-188172972-36-FFI-10min: index -1 is out of bounds for axis 0 with size 0
Error processing TIC-350559457-96-FFI-200s: index -1 is out of bounds for axis 0 with size 0
Error processing TIC-23325655-100-FFI-200s: index -1 is out of bounds for axis 0 with size 0
Error processing TIC-404012623-100-FFI-200s: index -1 is out of bounds for axis 0 with size 0
Error processing TIC-404012628-100-FFI-200s: index -1 is out of bounds for axis 0 with size 0
